# INFO
* 자기장센서 계측 Log를 통해, classification
* initialValueLog: 초기 Offset 값 계측 로그
* logData: 자석이 위치한 상태에서의 계측 로그


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyts.image import GramianAngularField
from keras.models import load_model

In [2]:
########## Global Variables ##########
sensorIdx = [4, 9, 14, 20, 25, 30]
sensorName = [f"s{i}" for i in np.arange(1, 7, 1)]
gaf = GramianAngularField(method= "difference")
colName = "col"
skipNumber = 300
useRow = 4000

In [ ]:
initialValueLog = pd.read_table("../data/initialValue.log", names= [colName])
initialValues = []
for i in range(initialValueLog.shape[0]):
    if len(initialValueLog.iloc[i, 0]) > 41 or len(initialValueLog.iloc[i, 0]) < 39:
        pass
    else:
        initialValues.append(initialValueLog.iloc[i, 0])
initialValues = np.array(initialValues)

initialSensors = []
for i in sensorIdx:
    initialSensors.append([s[i : i + 5] for s in initialValues])
initialSensors = pd.DataFrame(np.array(initialSensors, dtype= np.float32).T, columns= sensorName)

# meanInitialSensors = pd.DataFrame(initialSensors.mean(axis= 0), columns= ["init"])
meanInitialSensors = initialSensors.mean(axis= 0).to_numpy()

In [ ]:
logData = pd.read_table("../data/class21.log", names= [colName], skiprows= skipNumber, nrows= useRow)
data = []
for i in range(logData.shape[0]):
    if len(logData.iloc[i, 0]) > 41 or len(logData.iloc[i, 0]) < 39:
        pass
    else:
        data.append(logData.iloc[i, 0])
data = np.array(data)

sensorData = []
for i in sensorIdx:
    sensorData.append([d[i : i + 5] for d in data])
sensorData = np.array(sensorData, dtype= np.float32).T
sensorData.shape

(3999, 6)

In [42]:
calibrated = [sensorData[:, i] - meanInitialSensors[i] for i in range(6)]
calibrated = np.array(calibrated).reshape(-1, 6)
calibrated.shape

(3999, 6)

In [43]:
splited = []
for i in range(int(calibrated.shape[0] / 16)):
    splited.append(calibrated[i * 16 : 16 + (i * 16), :])
splited = np.array(splited)
splited.shape

(249, 16, 6)

In [44]:
encode = []
for i in range(splited.shape[0]):
    res = []
    for j in range(splited.shape[2]):
        s = splited[i, :, j].reshape(-1, 1)
        e = gaf.fit_transform(s.T).reshape(16, 16)
        res.append(e)
    res = np.array(res)
    encode.append(res)
encode = np.array(encode)
encode.shape

(249, 6, 16, 16)

In [45]:
reshaped = encode.reshape(encode.shape[0], 1, 16, 16, 6)

In [46]:
trainModel = load_model("../data/Test0919_6Channel_Epoch_100.h5")

predictResult = []
for i in range(reshaped.shape[0]):
    predictClass = np.argmax(trainModel.predict(reshaped[i]), axis= 1)
    predictResult.append(predictClass)

1/1 [==============================] - 0s 28ms/step


In [47]:
print(np.unique(predictResult, return_counts= True))

(array([ 9, 11, 15, 19, 21, 26], dtype=int64), array([  3,   2,   4,   1,  79, 160], dtype=int64))


# Data description
1. initialValue: 전방 초기값
2. class21: class 21정답 데이터
3. class25: class 25 위치 데이터
4. class25_2: class 25 - 3번 센서 근처